In [1]:
import pandas as pd
import os
import re
import string
import nltk
import logging
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from dotenv import load_dotenv

In [2]:
#Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

In [3]:
#Load environment variables
load_dotenv()

True

In [4]:
#Download necessary NLTK data
nltk_resources = ["stopwords", "punkt", "wordnet"]
for resource in nltk_resources:
    try:
        nltk.data.find(f"corpora/{resource}") if resource != "punkt" else nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/hachikaruanyakwee/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
#Define correct file paths
merged_dir = os.path.join("..", "data", "merged")
processed_dir = os.path.join("..", "data", "processed")
os.makedirs(processed_dir, exist_ok=True)

input_file = os.path.join(merged_dir, "merged_data.csv")
output_file = os.path.join(processed_dir, "cleaned_data.csv")

In [6]:
#Function to clean text
def clean_text(text):
    if not isinstance(text, str):
        return ""

    #Remove URLs and mentions
    text = re.sub(r"http\S+|www\S+|@\w+", "", text)

    #Remove emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  #emoticons
        "\U0001F300-\U0001F5FF"  #symbols & pictographs
        "\U0001F680-\U0001F6FF"  #transport & map symbols
        "\U0001F1E0-\U0001F1FF"  #flags (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    text = emoji_pattern.sub(r"", text)

    #Convert text to lowercase
    text = text.lower()

    #Expand contractions
    contractions = {
        "i'm": "i am", "you're": "you are", "he's": "he is", "she's": "she is",
        "it's": "it is", "we're": "we are", "they're": "they are", "can't": "cannot",
        "won't": "will not", "don't": "do not", "doesn't": "does not", "didn't": "did not"
    }
    for contraction, expanded in contractions.items():
        text = text.replace(contraction, expanded)

    #Remove punctuation and special characters
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text) # Remove any remaining non-alphanumeric characters

    #Tokenization
    words = word_tokenize(text)

    #Remove stop words and short words
    stop_words = set(stopwords.words("english"))
    words = [word for word in words if word not in stop_words and len(word) > 2]

    #Lemmatization
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]

    #Rejoin words into a cleaned string
    return " ".join(words)

In [7]:
#Load dataset
try:
    logging.info(f"Loading merged dataset from: {input_file}")
    df = pd.read_csv(input_file)

    #Check for required columns
    if "text" not in df.columns or "Source" not in df.columns:
        raise ValueError("Missing required columns ('text' and 'Source') in merged dataset!")
    logging.info(f"Dataset loaded successfully. Found {len(df)} records.")

    #Apply text preprocessing
    logging.info(f"Applying text preprocessing to {len(df)} records.")
    df["cleaned_text"] = df["text"].apply(clean_text)
    logging.info("Text preprocessing complete.")

    #Remove duplicates
    initial_rows = len(df)
    df.drop_duplicates(subset=["cleaned_text"], inplace=True)
    duplicates_removed = initial_rows - len(df)
    logging.info(f"Removed {duplicates_removed} duplicate rows based on 'cleaned_text'.")

    #Save cleaned dataset
    logging.info(f"Saving cleaned dataset to: {output_file}")
    df.to_csv(output_file, index=False)
    logging.info(f"✅ Text preprocessing complete! Cleaned data saved to: {output_file}")

except FileNotFoundError:
    logging.error(f"❌ Error: Merged dataset not found at {input_file}!")
except ValueError as ve:
    logging.error(f"❌ Error: {ve}")
except Exception as e:
    logging.error(f"❌ An unexpected error occurred: {e}")

2025-03-19 18:26:54,829 - INFO - Loading merged dataset from: ../data/merged/merged_data.csv
2025-03-19 18:26:54,850 - INFO - Dataset loaded successfully. Found 1598 records.
2025-03-19 18:26:54,852 - INFO - Applying text preprocessing to 1598 records.
2025-03-19 18:26:56,794 - INFO - Text preprocessing complete.
2025-03-19 18:26:56,797 - INFO - Removed 95 duplicate rows based on 'cleaned_text'.
2025-03-19 18:26:56,797 - INFO - Saving cleaned dataset to: ../data/processed/cleaned_data.csv
2025-03-19 18:26:56,802 - INFO - ✅ Text preprocessing complete! Cleaned data saved to: ../data/processed/cleaned_data.csv
